In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# For splitting data
from sklearn.model_selection import train_test_split

# For preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# For fitting
#from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import (accuracy_score, classification_report,
                           confusion_matrix, roc_curve, precision_recall_curve,
                           auc, average_precision_score)
from sklearn.pipeline import make_pipeline

# For improving Random forest
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

In [9]:
#Read preprocessed data
data = pd.read_csv("./preprocessed_data.csv", low_memory = False)

#Read updated data with feature interaction
data_transformed = pd.read_csv("./cj_data_numerical.csv", low_memory = False)

In [10]:
data_transformed.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,diag_1,diag_2,diag_3,max_glu_serum,A1Cresult,metformin,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,insulin,change,diabetesMed,readmitted,past_encounters,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,time_in_hospital num_lab_procedures,time_in_hospital num_procedures,time_in_hospital num_medications,time_in_hospital number_outpatient,time_in_hospital number_emergency,time_in_hospital number_inpatient,time_in_hospital number_diagnoses,num_lab_procedures num_procedures,num_lab_procedures num_medications,num_lab_procedures number_outpatient,num_lab_procedures number_emergency,num_lab_procedures number_inpatient,num_lab_procedures number_diagnoses,num_procedures num_medications,num_procedures number_outpatient,num_procedures number_emergency,num_procedures number_inpatient,num_procedures number_diagnoses,num_medications number_outpatient,num_medications number_emergency,num_medications number_inpatient,num_medications number_diagnoses,number_outpatient number_emergency,number_outpatient number_inpatient,number_outpatient number_diagnoses,number_emergency number_inpatient,number_emergency number_diagnoses,number_inpatient number_diagnoses
0,24437208,135,Caucasian,Female,[50-60),2,1,Diseases of the circulatory system,Injury and poisoning,Diseases of the digestive system,NaN,NaN,Steady,No,No,Change,No,No,Steady,Ch,Yes,1,0,8,77,6,33,0,0,0,8,616,48,264,0,0,0,64,462,2541,0,0,0,616,198,0,0,0,48,0,0,0,264,0,0,0,0,0,0
1,29758806,378,Caucasian,Female,[50-60),3,1,Diseases of the musculoskeletal system and con...,Mental disorders,"Endocrine, nutritional and metabolic diseases,...",NaN,NaN,No,No,No,No,No,No,No,No,No,0,0,2,49,1,11,0,0,0,3,98,2,22,0,0,0,6,49,539,0,0,0,147,11,0,0,0,3,0,0,0,33,0,0,0,0,0,0
2,189899286,729,Caucasian,Female,[80-90),1,3,Injury and poisoning,Diseases of the respiratory system,External causes of injury,NaN,>7,Steady,No,No,No,No,No,No,No,Yes,0,0,4,68,2,23,0,0,0,9,272,8,92,0,0,0,36,136,1564,0,0,0,612,46,0,0,0,18,0,0,0,207,0,0,0,0,0,0
3,64331490,774,Caucasian,Female,[80-90),1,1,"Endocrine, nutritional and metabolic diseases,...",Diseases of the circulatory system,Diseases of the circulatory system,NaN,>8,Steady,No,No,Steady,No,No,No,Ch,Yes,0,0,3,46,0,20,0,0,0,9,138,0,60,0,0,0,27,0,920,0,0,0,414,0,0,0,0,0,0,0,0,180,0,0,0,0,0,0
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,Diseases of the genitourinary system,Neoplasms,"Endocrine, nutritional and metabolic diseases,...",NaN,NaN,No,Steady,No,No,No,No,No,No,Yes,0,0,5,49,0,5,0,0,0,3,245,0,25,0,0,0,15,0,245,0,0,0,147,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0


In [13]:
#numeric_features = ['age', 'credit_amount', 'duration']
one_hot_cols = ["diag_1", "diag_2", "diag_3"]
ordinal_cols = ["race", "gender", "max_glu_serum", "A1Cresult", "insulin","change", "diabetesMed", "metformin", "glimepiride",
                "glipizide", "glyburide", "pioglitazone", "rosiglitazone", "age"
]
ordinal_categories = [
    ["Caucasian", "AfricanAmerican", "Hispanic", "Asian", "Other", "None"],
    ["Male", "Female", "None"],
    ["None", "Norm", ">200", ">300"],
    ["None", "Norm", ">7", ">8"],
    ["No", "Steady", "Change", "None"],
    ["No", "Ch", "None"],
    ["No", "Yes", "None"],
    ["No", "Steady", "Change", "None"],
    ["No", "Steady", "Change", "None"],
    ["No", "Steady", "Change", "None"],
    ["No", "Steady", "Change", "None"],
    ["No", "Steady", "Change", "None"],
    ["No", "Steady", "Change", "None"],
    ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
     "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)", "None"],
]

ohe = OneHotEncoder(drop=None, handle_unknown="ignore", sparse_output=False)
data_ohe = pd.DataFrame(
    ohe.fit_transform(data[one_hot_cols]),
    columns=ohe.get_feature_names_out(one_hot_cols),
    index=data.index
)

ordinal = OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1)
data_ord = pd.DataFrame(
    ordinal.fit_transform(data[ordinal_cols]),
    columns=ordinal_cols,
    index=data.index
)

data_remaining = data_transformed.drop(columns=one_hot_cols + ordinal_cols + ["encounter_id"] + ["patient_nbr"])

data_encoded = pd.concat([data_remaining, data_ohe, data_ord], axis=1)

data_encoded.head()

,admission_type_id,discharge_disposition_id,readmitted,past_encounters,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,time_in_hospital num_lab_procedures,time_in_hospital num_procedures,time_in_hospital num_medications,time_in_hospital number_outpatient,time_in_hospital number_emergency,time_in_hospital number_inpatient,time_in_hospital number_diagnoses,num_lab_procedures num_procedures,num_lab_procedures num_medications,num_lab_procedures number_outpatient,num_lab_procedures number_emergency,num_lab_procedures number_inpatient,num_lab_procedures number_diagnoses,num_procedures num_medications,num_procedures number_outpatient,num_procedures number_emergency,num_procedures number_inpatient,num_procedures number_diagnoses,num_medications number_outpatient,num_medications number_emergency,num_medications number_inpatient,num_medications number_diagnoses,number_outpatient number_emergency,number_outpatient number_inpatient,number_outpatient number_diagnoses,number_emergency number_inpatient,number_emergency number_diagnoses,number_inpatient number_diagnoses,"diag_1_Complications of pregnancy, childbirth, and the puerperium",diag_1_Congenital anomalies,diag_1_Diseases of the blood and blood-forming organs,diag_1_Diseases of the circulatory system,diag_1_Diseases of the digestive system,diag_1_Diseases of the genitourinary system,diag_1_Diseases of the musculoskeletal system and connective tissue,diag_1_Diseases of the nervous system and sense organs,diag_1_Diseases of the respiratory system,diag_1_Diseases of the skin and subcutaneous tissue,"diag_1_Endocrine, nutritional and metabolic diseases, and immunity disorders",diag_1_External causes of injury,diag_1_Infectious and parasitic diseases,diag_1_Injury and poisoning,diag_1_Mental disorders,diag_1_Neoplasms,diag_1_Supplemental classification,"diag_1_Symptoms, signs, and ill-defined conditions",diag_1_Unknown,"diag_2_Complications of pregnancy, childbirth, and the puerperium",diag_2_Congenital anomalies,diag_2_Diseases of the blood and blood-forming organs,diag_2_Diseases of the circulatory system,diag_2_Diseases of the digestive system,diag_2_Diseases of the genitourinary system,diag_2_Diseases of the musculoskeletal system and connective tissue,diag_2_Diseases of the nervous system and sense organs,diag_2_Diseases of the respiratory system,diag_2_Diseases of the skin and subcutaneous tissue,"diag_2_Endocrine, nutritional and metabolic diseases, and immunity disorders",diag_2_External causes of injury,diag_2_Infectious and parasitic diseases,diag_2_Injury and poisoning,diag_2_Mental disorders,diag_2_Neoplasms,diag_2_Supplemental classification,"diag_2_Symptoms, signs, and ill-defined conditions",diag_2_Unknown,"diag_3_Complications of pregnancy, childbirth, and the puerperium",diag_3_Congenital anomalies,diag_3_Diseases of the blood and blood-forming organs,diag_3_Diseases of the circulatory system,diag_3_Diseases of the digestive system,diag_3_Diseases of the genitourinary system,diag_3_Diseases of the musculoskeletal system and connective tissue,diag_3_Diseases of the nervous system and sense organs,diag_3_Diseases of the respiratory system,diag_3_Diseases of the skin and subcutaneous tissue,"diag_3_Endocrine, nutritional and metabolic diseases, and immunity disorders",diag_3_External causes of injury,diag_3_Infectious and parasitic diseases,diag_3_Injury and poisoning,diag_3_Mental disorders,diag_3_Neoplasms,diag_3_Supplemental classification,"diag_3_Symptoms, signs, and ill-defined conditions",diag_3_Unknown,race,gender,max_glu_serum,A1Cresult,insulin,change,diabetesMed,metformin,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,age
0,2,1,1,0,8,77,6,33,0,0,0,8,616,48,264,0,0,0,64,462,2541,0,0,0,616,198,0,0,0,48,0,0,0,264,0,0,0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0

In [14]:
X = data_encoded.drop(columns=["readmitted"])
y = data_encoded['readmitted']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [23]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight = 'balanced', max_samples = 0.8)

param_grid = {
    'n_estimators': [100,300, 500],        # number of trees
    'max_depth': [None, 5, 10, 15],        # tree depth
    #'min_samples_split': [5, 10],        # min samples to split a node
    #'min_samples_leaf': [2, 5, 10],          # min samples per leaf
    #'max_features': [0.3, 'sqrt']        # feature subset for splitting
}

In [24]:
scoring = make_scorer(roc_auc_score, needs_proba=True)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


In [25]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,              
    n_jobs=-1,          
    verbose=2,          
    scoring=scoring         
)

In [26]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


[CV] END ...................max_depth=None, n_estimators=100; total time=   9.7s
[CV] END ...................max_depth=None, n_estimators=100; total time=   9.7s
[CV] END ...................max_depth=None, n_estimators=100; total time=   9.8s
[CV] END ...................max_depth=None, n_estimators=100; total time=   9.8s
[CV] END ...................max_depth=None, n_estimators=100; total time=  10.0s
[CV] END ...................max_depth=None, n_estimators=300; total time=  30.4s
[CV] END ...................max_depth=None, n_estimators=300; total time=  31.2s
[CV] END ...................max_depth=None, n_estimators=300; total time=  31.5s
[CV] END ......................max_depth=5, n_estimators=100; total time=   2.9s
[CV] END ......................max_depth=5, n_estimators=100; total time=   3.2s
[CV] END ......................max_depth=5, n_estimators=100; total time=   3.0s
[CV] END ...................max_depth=None, n_estimators=300; total time=  32.3s
[CV] END ...................

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END .....................max_depth=10, n_estimators=300; total time=  14.5s
[CV] END .....................max_depth=10, n_estimators=300; total time=  14.6s
[CV] END .....................max_depth=10, n_estimators=300; total time=  14.4s
[CV] END .....................max_depth=10, n_estimators=300; total time=  14.1s
[CV] END .....................max_depth=10, n_estimators=300; total time=  14.1s
[CV] END .....................max_depth=15, n_estimators=100; total time=   6.4s
[CV] END .....................max_depth=15, n_estimators=100; total time=   6.7s
[CV] END .....................max_depth=10, n_estimators=500; total time=  23.0s
[CV] END .....................max_depth=15, n_estimators=100; total time=   7.0s
[CV] END .....................max_depth=15, n_estimators=100; total time=   7.2s
[CV] END .....................max_depth=15, n_estimators=100; total time=   7.3s
[CV] END .....................max_depth=10, n_estimators=500; total time=  23.0s
[CV] END ...................

GridSearchCV(estimator=RandomForestClassifier(class_weight='balanced',
                                              max_samples=0.8, n_jobs=-1,
                                              random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 15],
                         'n_estimators': [100, 300, 500]},
             scoring=make_scorer(roc_auc_score, response_method='predict_proba'),
             verbose=2)

In [27]:
best_rf = grid_search.best_estimator_

In [28]:
# Probabilities for the positive class
y_pred_proba = best_rf.predict_proba(X_train)[:, 1]

# Compute AUC
auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC-ROC:", auc_score)

Train AUC-ROC: 0.6482668963579745


In [29]:


# Probabilities for the positive class
y_pred_proba = best_rf.predict_proba(X_test)[:, 1]

# Compute AUC
auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC-ROC:", auc_score)

Test AUC-ROC: 0.6314861866432715


In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X.columns)         
print(importances.sort_values(ascending=False).head(15))

discharge_disposition_id               0.166678
num_lab_procedures number_inpatient    0.070252
time_in_hospital number_inpatient      0.069765
number_inpatient number_diagnoses      0.060033
num_medications number_inpatient       0.056572
number_inpatient                       0.048843
time_in_hospital number_diagnoses      0.048686
time_in_hospital num_lab_procedures    0.037125
time_in_hospital num_medications       0.036832
age                                    0.033739
num_medications number_diagnoses       0.031300
num_lab_procedures number_diagnoses    0.029536
time_in_hospital                       0.026810
num_lab_procedures num_medications     0.025770
num_procedures number_inpatient        0.022249
dtype: float64


In [7]:
total_cols = one_hot_cols + ordinal_cols
data_transformed_cols = data_transformed.columns
numerical_cols = [col for col in data_transformed_cols if col not in total_cols]

In [8]:
numerical_cols 

['encounter_id',
 'patient_nbr',
 'admission_type_id',
 'discharge_disposition_id',
 'readmitted',
 'past_encounters',
 'time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'number_diagnoses',
 'time_in_hospital num_lab_procedures',
 'time_in_hospital num_procedures',
 'time_in_hospital num_medications',
 'time_in_hospital number_outpatient',
 'time_in_hospital number_emergency',
 'time_in_hospital number_inpatient',
 'time_in_hospital number_diagnoses',
 'num_lab_procedures num_procedures',
 'num_lab_procedures num_medications',
 'num_lab_procedures number_outpatient',
 'num_lab_procedures number_emergency',
 'num_lab_procedures number_inpatient',
 'num_lab_procedures number_diagnoses',
 'num_procedures num_medications',
 'num_procedures number_outpatient',
 'num_procedures number_emergency',
 'num_procedures number_inpatient',
 'num_procedures number_diagnoses',
 'num_medications number_outp